# Step 1: tune the hyperparameters

Runs the lab's Hyperband search (`SeqAdage.tune_model`) and saves the best hyperparameters to
`runs/tuning/best_hparams.json`, which the training notebook reads. Takes about 2 hours; skipped if already done.

Settings, and the lab's code. The search ranges are the ones in thesis Table 1.

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's startup messages

PROJECT = os.path.expandvars("$SCRATCH/CSC-488-Code")    # project folder: data, lab code and results
DATA_PATH = os.path.join(PROJECT, "data", "Copy of ecoli_MG1655plus_genome_part2_log_counts_norm_01.csv")
SEQADAGE = os.path.join(PROJECT, "seqADAGE")              # the lab's code (my fork, Jordin branch)
RUN_DIR = os.path.join(PROJECT, "runs")                   # every notebook saves and reads its results here
os.chdir(PROJECT)

import json
import types
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(SEQADAGE, "Py"))
from adage import SeqAdage

TUNE_SEED = 960
MAX_EPOCHS = 50  # the longest any setting is trained
# [min, max, step] or a list of choices. The number of nodes and both activations are fixed.
TUNE_RANGES = dict(u=[450, 450, 1], d=[0.0, 0.5, 0.1], a1=["tanh"], a2=["tanh"],
                   i=["glorot_uniform", "glorot_normal"], k1=[0, 2e-7, 2e-8], k2=[0, 1e-5, 1e-6],
                   l=[0.001, 0.5, 0.01], b=[5, 100, 5], m=[0.0, 0.9, 0.1])
FORCE = False  # True = search again even if best_hparams.json exists

`tune_model` saves its search to a fixed folder and ranks settings by the full loss, which includes the weight penalties.
This points it at my run folder instead and ranks settings by the reconstruction error on the held-out samples alone
(`val_mse`). The penalties still apply during training.

In [ ]:
def redirect_tuner(SeqAdage, directory, max_epochs):
    import keras
    import keras_tuner

    def hyperband(**kwargs):
        kwargs.update(directory=directory, max_epochs=max_epochs)
        hypermodel = kwargs["hypermodel"]
        lab_build = hypermodel.build

        def build_with_mse(hp):
            model = lab_build(hp)  # the lab's model, compiled with its optimizer and loss
            model.compile(optimizer=model.optimizer, loss=model.loss,
                          metrics=[keras.metrics.MeanSquaredError(name="mse")])  # error without the penalties
            return model

        hypermodel.build = build_with_mse
        kwargs["objective"] = keras_tuner.Objective("val_mse", direction="min")
        return keras_tuner.Hyperband(**kwargs)

    # only changes the keras_tuner that the SeqAdage module sees
    SeqAdage.keras_tuner = types.SimpleNamespace(HyperParameters=keras_tuner.HyperParameters, Hyperband=hyperband)


def trials_table(tuner):
    """One row per setting tried: its hyperparameters, score and rank."""
    trials = tuner.oracle.get_best_trials(num_trials=len(tuner.oracle.trials))
    return pd.DataFrame([t.hyperparameters.get_config()["values"] | {"score": t.score, "rank": rank}
                         for rank, t in enumerate(trials)])

Run the search, then save the best hyperparameters and a table of every setting tried.

In [ ]:
import keras

tune_dir = os.path.join(RUN_DIR, "tuning")
best_path = os.path.join(tune_dir, "best_hparams.json")
os.makedirs(tune_dir, exist_ok=True)
if os.path.exists(best_path) and not FORCE:
    print("already tuned, best hyperparameters:", open(best_path).read())
else:
    redirect_tuner(SeqAdage, tune_dir, MAX_EPOCHS)
    keras.utils.set_random_seed(TUNE_SEED)  # tune_model adds its noise before it sets its own seed
    sa = SeqAdage.SeqAdage(DATA_PATH, seed=TUNE_SEED, v=1)
    sa.input_file = "compendium.csv"  # tune_model names its search folder after this
    best_hps, tuner = sa.tune_model(TUNE_SEED, **TUNE_RANGES)

    best = {k: best_hps[0].values[k] for k in ["units", "dropout", "act1", "act2", "init", "kl1", "kl2", "lr", "bs", "mm"]}
    best["lr"], best["kl1"], best["kl2"] = round(best["lr"], 6), round(best["kl1"], 10), round(best["kl2"], 9)
    with open(best_path, "w") as f:
        json.dump(best, f, indent=2)
    trials = trials_table(tuner)
    trials.to_csv(os.path.join(tune_dir, "trials.csv"), index=False)
    print(f"{len(trials)} settings tried, best score {trials['score'].min():.5f}")
    print("best hyperparameters:", json.dumps(best, indent=2))